# Sky Background

This notebook shows how Fig. 6 in the PlatoSim3 paper was created.

### Setup notebook

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

### Imports

In [ ]:
import os
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt
from matplotlib import colors
from astropy import units as u
from astropy.coordinates import SkyCoord

# PlatoSim
import platosim.utilities  as ut
from platosim.h5           import h5get
from platosim.simulation   import Simulation
from platosim.validation   import switchOffAllEffects
from platosim.matplotlibrc import setup_paper
setup_paper()

## All-sky projection

In [ ]:
# Generate a grid (N=130 is needed for complete Galactic grid)
N = 130
x = np.linspace(0, 360, N)
y = np.linspace(-90, 90, N)
x, y = np.meshgrid(x, y)
RA  = x.flatten() * u.degree
Dec = y.flatten() * u.degree

In [ ]:
# Grid in Equatorial coordinates
c = SkyCoord(ra=RA, dec=Dec, frame='icrs')
ra_rad = c.ra.wrap_at(180 * u.deg).radian
dec_rad = c.dec.radian
RA, Dec = RA.value, Dec.value

plt.figure(figsize=(8,4.2))
plt.subplot(111, projection="aitoff")
plt.title("Aitoff Equatorial", pad=25)
plt.grid(True)
plt.plot(ra_rad, dec_rad, 'o', markersize=1)
plt.subplots_adjust(top=0.95,bottom=0.0)
plt.show()

In [ ]:
# Compared to the Galactic coordinate system
eli = c.geocentricmeanecliptic
l = -eli.lon.wrap_at("180d").radian
b = eli.lat.radian

plt.figure(figsize=(9,5))
plt.subplot(111, projection="aitoff")
plt.title("Aitoff Ecliptic", pad=25)
plt.grid(True)
plt.scatter(l, b, s=2)
plt.subplots_adjust(top=0.95,bottom=0.0)
plt.show()

In [ ]:
# Compared to the Galactic coordinate system
gal = c.galactic
l = -gal.l.wrap_at("180d").radian
b = gal.b.radian

plt.figure(figsize=(9,5))
plt.subplot(111, projection="aitoff")
plt.title("Aitoff Galactic", pad=25)
plt.grid(True)
plt.scatter(l, b, s=2)
plt.subplots_adjust(top=0.95,bottom=0.0)
plt.show()

In [ ]:
# Compute the sky background for each grid point as reported by PlatoSim
# Note this simulation takes several hours using N = 130 !
N = len(RA)
sky = np.zeros(N)

for i, ra, dec in zip(tqdm(range(N), bar_format=ut.tqdmBar()), RA, Dec):
    
    # Create simulation object
    sim = Simulation("output_skybackground", outputDir=os.getcwd())
    
    # Switch off all effects (Note: works here because we only simulate 1 pixel)
    switchOffAllEffects(sim)
    
    # Turn off all savings
    sim.turnOffAllOutput()
    
    # Simulate a single pixel
    sim["ObservingParameters/NumExposures"] = 1
    sim["ObservingParameters/RApointing"]   = ra
    sim["ObservingParameters/DecPointing"]  = dec
    sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'yes'
    sim["Sky/SkyBackground/BackgroundValue"]          = -1       
    sim["SubField/NumRows"]    = 1
    sim["SubField/NumColumns"] = 1
    
    # Define catalogue and set it
    if i == 0:
        row = np.array([])
        col = np.array([])
        mag = np.array([])
        ID  = []
        starcatFile = os.getcwd() + "/starcat_background.txt"
        sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)
    else:
        sim["ObservingParameters/StarCatalogFile"] = starcatFile 
    
    # Set subfield to same location as platform pointing
    sim.setSubfieldAroundCoordinates(np.deg2rad(ra), np.deg2rad(dec), 1, 1, normal=True)
    
    # Simulate each sky background
    simfile = sim.run(removeOutputFile = True)
    sky[i] = simfile.getBackgroundValue()
    
# Save the sky background
np.savetxt(f"{os.getcwd()}/skybackground.txt", np.transpose([RA, Dec, sky]))

In [ ]:
# Load simulation
data = np.loadtxt(f"{os.getcwd()}/skybackground.txt")
RA, Dec, sky = data[:,0], data[:,1], data[:,2]
equa = SkyCoord(RA, Dec, frame='icrs', unit=u.deg)

# Pointing of platform
# sim = Simulation("output_skybackground", outputDir=os.getcwd())
# platformRA  = sim['ObservingParameters/RApointing']  * u.degree
# platformDec = sim['ObservingParameters/DecPointing'] * u.degree
raLOPN, decLOPN  = 277.18023 * u.degree, 52.85952 * u.degree
raLOPS, decLOPS  = 93.49134 * u.degree, -42.93544 * u.degree 
raNPF,  decNPF   = 265.08002279 * u.degree, 39.5836954 * u.degree
raSPF,  decSPF   = 86.79870508 * u.degree, -46.39594703 * u.degree
platformLOPN  = SkyCoord(ra=raLOPN, dec=decLOPN, frame='icrs')
platformLOPS  = SkyCoord(ra=raLOPS, dec=decLOPS, frame='icrs')
platformNPF   = SkyCoord(ra=raNPF,  dec=decNPF,  frame='icrs')
platformSPF   = SkyCoord(ra=raSPF,  dec=decSPF,  frame='icrs')

# SELCT SKY FRAME BELOW (UNCOMMENT ONLY ONE BLOCK!)

# Equatorial
# x = -equa.ra.wrap_at('180d').radian
# y = equa.dec.radian
# xlab = r'RA, $\alpha$ [deg]'
# ylab = r'Dec, $\delta$ [deg]'
# #----- Ecliptic line ------
# lon = np.deg2rad(np.linspace(181, 540, 100))
# lat = np.zeros_like(lon)
# ecli = SkyCoord(frame="geocentricmeanecliptic", lon=lon, lat=lat, unit=u.radian)
# equa_line = ecli.transform_to("icrs")
# x_line = -equa_line.ra.wrap_at('180d').radian
# y_line = equa_line.dec.radian
# #----- Pointing --------
# x_platform = -platform.ra.wrap_at('180d').radian
# y_platform = platform.dec.radian

# Ecliptic
# grid = equa.geocentricmeanecliptic #barycentricmeanecliptic
# x = -grid.lon.wrap_at('180d').radian
# y = grid.lat.radian
# xlab = r'Longitude, $\lambda$ [$^{\circ}$]'
# ylab = r'Latitude, $\beta$ [$^{\circ}$]'
# #----- Pointing --------
# platformLOPN = platformLOPN.geocentricmeanecliptic
# platformLOPS = platformLOPS.geocentricmeanecliptic
# platformNPF  = platformNPF.geocentricmeanecliptic
# platformSPF  = platformSPF.geocentricmeanecliptic
# xLOPN, yLOPN = -platformLOPN.lon.wrap_at('180d').radian, platformLOPN.lat.radian
# xLOPS, yLOPS = -platformLOPS.lon.wrap_at('180d').radian, platformLOPS.lat.radian
# xNPF, yNPF = -platformNPF.lon.wrap_at('180d').radian, platformNPF.lat.radian
# xSPF, ySPF = -platformSPF.lon.wrap_at('180d').radian, platformSPF.lat.radian

# Galactic
grid = equa.galactic
x = -grid.l.wrap_at('180d').radian
y = grid.b.radian
xlab = r'Longitude, $l$ [deg]'
ylab = r'Latitude, $b$ [deg]'
#----- Ecliptic line ------
lon = np.deg2rad(np.linspace(80, 435, 100))
lat = np.zeros_like(lon)
ecli = SkyCoord(frame="geocentricmeanecliptic", lon=lon, lat=lat, unit=u.radian)
line = ecli.galactic
x_line = -line.l.wrap_at('180d').radian
y_line = line.b.radian
#----- Pointing --------
platformLOPN = platformLOPN.galactic
platformLOPS = platformLOPS.galactic
platformNPF  = platformNPF.galactic
platformSPF  = platformSPF.galactic
xLOPN, yLOPN = -platformLOPN.l.wrap_at('180d').radian, platformLOPN.b.radian
xLOPS, yLOPS = -platformLOPS.l.wrap_at('180d').radian, platformLOPS.b.radian
xNPF, yNPF = -platformNPF.l.wrap_at('180d').radian, platformNPF.b.radian
xSPF, ySPF = -platformSPF.l.wrap_at('180d').radian, platformSPF.b.radian

In [ ]:
# Add the sky projection ontop as transparent layer
fig, ax = plt.subplots(figsize=(10,7))
axes = fig.add_subplot(111, projection='aitoff', facecolor='none')

# Plot Aitoff projection in Galactic coordinates
fs, ms = 20, 100

# Plot the targets on the sky (autumn_r, rainbow)
im = plt.scatter(x, y, c=sky, s=ms, cmap="cubehelix", norm=colors.LogNorm(), zorder=0)
plt.plot(x_line, y_line, "--", c="royalblue", lw=1.5, zorder=1)
plt.plot(xLOPN, yLOPN, 'X', c='orange',  ms=10)
plt.plot(xLOPS, yLOPS, 'X', c='magenta', ms=10)
# plt.plot(xNPF, yNPF, 'x', c='orange',  ms=10)
# plt.plot(xSPF, ySPF, 'x', c='magenta', ms=10)

# Vertical or horizontal colorbar showing magnitudes
cbarax = fig.add_axes([0.25, 0.1, 0.525, 0.03]) #([0.25, -0.05, 0.52, 0.03])
sky_min = sorted(sky)[1]
sky_max = np.max(sky)
cbar = plt.colorbar(im, orientation='horizontal', cax=cbarax, extend='both')
cbar.set_label(r'Sky background flux, $F_{\rm sky}$ [$\gamma \ s^{-1} \ \rm pixel^{-1}$]', fontsize=fs)
cbar.ax.tick_params(labelsize=fs)

# Change the tick labels so that they are 0->360, rather than -180->+180
tickLabels = np.array([150, 120, 90, 60, 30, 0, 330, 300, 270, 240, 210])
axes.set_xticklabels(tickLabels)

# Change y ticks and remove last to make space for title
tickLabels = np.array([-75, -60, -45, -30, -15, 0, 15, 30, 45, 60, 75])
axes.set_yticklabels(tickLabels)

# Plot ticks on top of data 
axes.tick_params(axis='both', which='major', labelsize=15, zorder=1)
axes.tick_params(axis='x', colors='w')

# Increase x and y tick labels
axes.xaxis.set_tick_params(labelsize=fs+1)
axes.yaxis.set_tick_params(labelsize=fs)

# Add axes labels
axes.set_xlabel(xlab)
axes.set_ylabel(ylab)
axes.xaxis.set_label_coords(0.52, -0.025)

# Set grid and remore outer ticks (if set by default)
axes.grid(True, alpha=0.3)
ax.axis('off')
plt.draw()
# plt.tight_layout();

# Save figure
fig.savefig('skybackground.png', bbox_inches='tight', dpi=200);

## Full-frame image

In [ ]:
outputFileName = "output_background_image"

In [ ]:
# Initialise PlatoSim
sim = Simulation(outputFileName, outputDir=os.getcwd())
    
# Turn off all savings
sim.turnOffAllOutput()
sim["ControlHDF5Content/WriteVariableBackgroundMap"] = True

# Observation
sim["ObservingParameters/NumExposures"] = 1

# Sky
sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'no'
sim["Sky/SkyBackground/BackgroundValue"]          = -1

# Subfield
sim["SubField/NumColumns"]      = 4510
sim["SubField/NumRows"]         = 4510
sim["SubField/ZeroPointColumn"] = 0
sim["SubField/ZeroPointRow"]    = 0

# Run simulation
sim.run(removeOutputFile=True);

In [ ]:
f = SimFile(outputFileName + ".hdf5")
bg = f.getBackgroundMap()[0]

In [ ]:
f.showMap(bg, figsize=(9,8), clabel='Photons/s/pixel');